## OpenRadioss2PhysicsNEMO Dataset Generation

This notebook automates sequential OpenRadioss simulations for generating training datasets for PhysicsNeMo machine learning models.

## Prerequisites:
1. Upload your `Bumper_Beam_AP_meshed_0000.rad` (starter file) and `Bumper_Beam_AP_meshed_0001.rad` (engine file) to Google Drive under: `MyDrive/OpenRadioss_Simulations/`
2. Run all cells sequentially
3. All outputs are saved directly to Google Drive for persistence

## Cell 2: Download and Setup OpenRadioss

In [3]:
# Download OpenRadioss Linux binaries
!curl -L -o OpenRadioss_win64.zip https://github.com/OpenRadioss/OpenRadioss/releases/download/latest-20260319/OpenRadioss_win64.zip

# Extract OpenRadioss
!tar -xf OpenRadioss_win64.zip

print("OpenRadioss downloaded and extracted successfully!")

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
  0      0   0      0   0      0      0      0                              0
  0      0   0      0   0      0      0      0                              0

  0      0   0      0   0      0      0      0                              0
  8 323.5M   8 27.17M   0      0 17.63M      0   00:18   00:01   00:17 27.17M
 15 323.5M  15 50.00M   0      0 19.58M      0   00:16   00:02   00:14 24.85M
 22 323.5M  22 72.07M   0      0 20.28M      0   00:15   00:03   00:12 23.93M
 31 323.5M  31 101.4M   0      0 22.29M      0   00:14   00:04   00:10 25.30M
 37 323.5M  37 120.7M   0      0 21.76M      0   00:14   00:05   00:09 24.11M
 47 323.5M  47 152.4M   0      0 23.26M      0   00:13   00:06   00:07 25.00M
 57 323.5M  57 185.1M   0      0 24.53M      0   00:13   00:07

OpenRadioss downloaded and extracted successfully!


In [7]:
!pip install lasso-python
!pip install tqdm


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\hmouradi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\hmouradi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Cell 4: Install Vortex-Radioss Dependencies to convert Radioss result (.ANIM) to D3PLOT

In [15]:
import sys

# Clone Vortex-Radioss repository

import sys
import shutil
from pathlib import Path
import subprocess

base_dir = Path.cwd()
repo_root = base_dir / "Vortex-Radioss"

# Clone repo
subprocess.run([
    "git", "clone",
    "https://github.com/Vortex-CAE/Vortex-Radioss",
    str(repo_root)
], check=True)

# Add repo to path
sys.path.insert(0, str(repo_root.resolve()))
shutil.move('./Vortex-Radioss/vortex_radioss', './')
try:
    from vortex_radioss.animtod3plot.Anim_to_D3plot import readAndConvert
    print("Vortex-Radioss imported successfully!")
except ImportError as e:
    print(f"Import error: {e}")
    print("Attempting alternative import path...")
    sys.path.insert(0, '/Vortex-Radioss/vortex_radioss')
    from animtod3plot.Anim_to_D3plot import readAndConvert
    print("Vortex-Radioss imported successfully via alternative path!")

Vortex-Radioss imported successfully!


## Cell 5: Create .bat Run Script

In [17]:
# Create Windows .bat run script
runradioss_bat_content = '''echo off

set OPENRADIOSS_PATH=%1
set sim_path=%2
set sim_file_name=%3
set number_of_threads=%4
set RAD_CFG_PATH=%OPENRADIOSS_PATH%\hm_cfg_files
set RAD_H3D_PATH=%OPENRADIOSS_PATH%\extlib\h3d\lib\win64
set PATH=%OPENRADIOSS_PATH%\extlib\hm_reader\win64;%PATH%
set PATH=%OPENRADIOSS_PATH%\extlib\intelOneAPI_runtime\win64;%PATH%
set KMP_STACKSIZE=400m
cd %sim_path%
%OPENRADIOSS_PATH%\exec\starter_win64.exe -i %sim_path%\%sim_file_name%_0000.rad -nt %number_of_threads% -np 1
%OPENRADIOSS_PATH%\exec\engine_win64_sp.exe -i %sim_path%\%sim_file_name%_0001.rad -nt %number_of_threads% 
'''

# Write the script
with open('./runradioss.bat', 'w') as f:
    f.write(runradioss_bat_content)


print(".bat run script created at ./runradioss.bat")

.bat run script created at ./runradioss.bat


## Cell 6: Main Simulation Script

In [2]:
import psutil

# Physical cores (CPU count)
physical = psutil.cpu_count(logical=False)

# Logical cores (Thread count)
logical = psutil.cpu_count(logical=True)

print(f"Physical Cores: {physical}")
print(f"Total Threads:  {logical}")

Physical Cores: 6
Total Threads:  12


In [19]:
import sys
import subprocess
import os
import random
from vortex_radioss.animtod3plot.Anim_to_D3plot import readAndConvert


def format_radioss_field(value):
    try:
        formatted = "{:10.2f}".format(value)
        if len(formatted) > 10:
            formatted = "{:.2E}".format(value).replace('E', 'E+').replace('E+-', 'E-')
            if len(formatted) > 10:
                formatted = "{:.1E}".format(value).replace('E', 'E+').replace('E+-', 'E-')

        return formatted.rjust(10)
    except ValueError:
        return "        "

def modify_radioss_starter(input_filepath, output_filepath, prop_changes, pole_y_offset=None):
    """
    Modify multiple PROP/SHELL thicknesses and optionally pole Y position in a RADIOSS starter file.

    Args:
        input_filepath: Path to the input starter file
        output_filepath: Path to the output starter file
        prop_changes: Dictionary mapping prop_id to new_thickness, e.g., {2: 1.5, 3: 2.0}
        pole_y_offset: Optional float to add to the YM and Y_M1 values of /RWALL/CYL/1
    """
    try:
        if not os.path.exists(input_filepath):
            print(f"Error: Input file not found at '{input_filepath}'")
            return

        with open(input_filepath, 'r') as infile, open(output_filepath, 'w') as outfile:

            in_prop_shell = False
            in_rwall = False
            rwall_line_counter = 0
            modifications_made = {}
            prop_id_str = "9999999"
            line_counter = 0
            modified_line = ""

            for line in infile:
                # Check if this line should be skipped (will be replaced with modified version)
                skip_line = False

                # Handle /PROP/SHELL modifications
                if line.strip().startswith('/PROP/SHELL/'):
                    in_prop_shell = True
                    prop_id_str = line[12:].strip()

                current_prop_id = int(prop_id_str)
                if current_prop_id in prop_changes:
                    if line_counter <= 8 and in_prop_shell:
                        line_counter += 1
                        in_prop_shell = True
                    else:
                        line_counter = 0
                        in_prop_shell = False

                    if line_counter == 8:
                        line_counter = 0
                        in_prop_shell = False
                        new_thickness = prop_changes[current_prop_id]
                        modified_line = line[:30] + format_radioss_field(new_thickness) + line[40:]
                        outfile.write(modified_line)
                        modifications_made[current_prop_id] = modifications_made.get(current_prop_id, 0) + 1
                        skip_line = True
                else:
                    if in_prop_shell:
                        line_counter = 0
                        in_prop_shell = False

                # Handle /RWALL/CYL/1 modifications
                if line.strip().startswith('/RWALL/CYL/1'):
                    in_rwall = True
                    rwall_line_counter = 0
                elif in_rwall:
                    rwall_line_counter += 1

                    # Line 7: YM data line (columns 31-40, index 30-40)
                    if rwall_line_counter == 7 and pole_y_offset is not None:
                        # Parse the YM value from columns 31-40
                        ym_str = line[30:40].strip()
                        try:
                            ym_val = float(ym_str)
                            new_ym = ym_val + pole_y_offset
                            modified_line = line[:30] + format_radioss_field(new_ym) + line[40:]
                            outfile.write(modified_line)
                            modifications_made['pole_YM'] = True
                            skip_line = True
                        except ValueError:
                            pass  # Will write original line below
                    # Line 9: Y_M1 data line (columns 31-40, index 30-40)
                    elif rwall_line_counter == 9 and pole_y_offset is not None:
                        # Parse the Y_M1 value from columns 31-40
                        ym1_str = line[30:40].strip()
                        try:
                            ym1_val = float(ym1_str)
                            new_ym1 = ym1_val + pole_y_offset
                            modified_line = line[:30] + format_radioss_field(new_ym1) + line[40:]
                            outfile.write(modified_line)
                            modifications_made['pole_Y_M1'] = True
                            skip_line = True
                        except ValueError:
                            pass  # Will write original line below
                    elif rwall_line_counter >= 10:
                        in_rwall = False
                        rwall_line_counter = 0

                # Write the line if it wasn't skipped
                if not skip_line:
                    outfile.write(line)

            for prop_id in prop_changes:
                if prop_id not in modifications_made:
                    print(f"Warning: /PROP/SHELL card with ID {prop_id} was not found or updated.")

            if pole_y_offset is not None:
                if 'pole_YM' not in modifications_made:
                    print(f"Warning: Pole YM position was not found or updated.")
                if 'pole_Y_M1' not in modifications_made:
                    print(f"Warning: Pole Y_M1 position was not found or updated.")

            print(f"Run saved to: '{output_filepath}'")

    except FileNotFoundError:
        print(f"Error: Could not open one of the files.")
    except Exception as e:
        print(f"Oops! you have to deal with this: {e}")

def copy_engine_file(master_engine_file_with_path, new_engine_file_with_path):

    try:
        if not os.path.exists(master_engine_file_with_path):
            print(f"Error: Input file not found at '{master_engine_file_with_path}'")
            return

        with open(master_engine_file_with_path, 'r') as infile, open(new_engine_file_with_path, 'w') as outfile:

            for line in infile:
                    outfile.write(line)
    except FileNotFoundError:
        print(f"Error: Could not open one of the files.")
    except Exception as e:
        print(f"Oops! you have to deal with this: {e}")

def run_radioss(openradiosspath:str, simpath:str, simfile:str, nt:int):
    # Use Linux shell script instead of Windows batch file
    bat_file_directory = "./runradioss.bat"
    
    # 1. Assemble the command as a list (this is safer than a single string)
    os.system("runradioss.bat" + " " + openradiosspath + " " + simpath + " " + simfile + " " + str(nt))


def convert_anim_to_d3plot(resultfile_dir: str, resultfile: str):
    file_stem = resultfile_dir + "/" + resultfile
    readAndConvert(file_stem, use_shell_mask=False, silent=False)


if __name__ == "__main__":

    print("GENERATING D3PLOT DATASET FOR PhysicsNemo")
    print("       POWERED BY OpenRadioss")
    ###########################################################
    # Number of threads of your processor
    nt = logical
    # Master Radioss input path
    master_input_path="./Bumper_Beam_Radioss_Input"
    # Path where raw_data will be saved
    sim_path = "./"
    # OpenRadioss path (installed in Colab environment)
    openradioss_path =  "./OpenRadioss"

    sim_name = "Bumper_Beam_AP_meshed"

    # Thickness range (min, max) - same for all props
    thickness_range = (1.2, 2.0)

    # List of PROP/SHELL IDs to modify with the same thickness
    prop_ids_to_change = [2, 7]

    # Pole Y position offset range (min, max)
    pole_y_range = (-500, 500)

    # Number of simulation runs
    number_of_sims = 3

    # The generated data structure:
    # ├── Run100/
    # │       ├── sim_name.d3plot
    # │       ├── sim_name_0000.rad     # RADIOSS STARTER FILE
    # │       └── sim_name_0001.rad     # RADIOSS ENGINE FILE
    # ├── Run101/
    # │       ├── sim_name.d3plot
    # │       ├── sim_name_0000.rad
    # │       └── sim_name_0001.rad
    # └── ...
    ###########################################################

    # Create RAW_DATA directory
    raw_data_path = os.path.join(sim_path, "RAW_DATA")
    os.makedirs(raw_data_path, exist_ok=True)

    # Check if master files exist
    master_starter_file = os.path.join( os.getcwd() , "Bumper_Beam_Radioss_Input" , sim_name + "_0000.rad")
    master_engine_file = os.path.join( os.getcwd() , "Bumper_Beam_Radioss_Input" , sim_name + "_0001.rad")

    if not os.path.exists(master_starter_file):
        print(f"ERROR: Master starter file not found: {master_starter_file}")
        sys.exit(1)

    if not os.path.exists(master_engine_file):
        print(f"ERROR: Master engine file not found: {master_engine_file}")
        sys.exit(1)

    print(f"Starting {number_of_sims} simulations...")
    print(f"Results will be saved to: {raw_data_path}")

    for i in range(number_of_sims):

        # Create run directory
        if i < 10:
            run_dir = os.path.join(raw_data_path, f"Run10{i}")
        else:
            run_dir = os.path.join(raw_data_path, f"Run1{i}")

        os.makedirs(run_dir, exist_ok=True)
        print(f"\n{'='*60}")
        print(f"Run {i+1}/{number_of_sims}: {run_dir}")
        print(f"{'='*60}")

        # Generate one random thickness for all props
        new_thickness = random.uniform(thickness_range[0], thickness_range[1])
        prop_changes = {prop_id: new_thickness for prop_id in prop_ids_to_change}
        print(f"Generated thickness: {new_thickness:.4f} mm for props: {prop_ids_to_change}")

        # Generate random pole Y offset
        pole_y_offset = random.uniform(pole_y_range[0], pole_y_range[1])
        print(f"Generated pole Y offset: {pole_y_offset:.2f} mm")

        new_starter_file = os.path.join(run_dir, sim_name + "_0000.rad")
        new_engine_file = os.path.join(run_dir, sim_name + "_0001.rad")

        modify_radioss_starter(master_starter_file, new_starter_file, prop_changes, pole_y_offset)
        copy_engine_file(master_engine_file, new_engine_file)

        # Run OpenRadioss simulation
        run_radioss(openradioss_path, run_dir, sim_name, nt)

        # Convert ANIM to D3PLOT format
        convert_anim_to_d3plot(run_dir, sim_name)

        print(f"Run {i+1} completed!")

    print(f"\n{'='*60}")
    print("ALL SIMULATIONS COMPLETED!")
    print(f"Dataset saved to: {raw_data_path}")
    print(f"{'='*60}")

GENERATING D3PLOT DATASET FOR PhysicsNemo
       POWERED BY OpenRadioss
Starting 3 simulations...
Results will be saved to: ./RAW_DATA

Run 1/3: ./RAW_DATA\Run100
Generated thickness: 1.3435 mm for props: [2, 7]
Generated pole Y offset: 214.91 mm
Run saved to: './RAW_DATA\Run100\Bumper_Beam_AP_meshed_0000.rad'
Mapping database
No files found..
Please check file stem:
e.g
C:/Folder/ModelA00*
Would be:
C:/Folder/Model*
Run 1 completed!

Run 2/3: ./RAW_DATA\Run101
Generated thickness: 1.6257 mm for props: [2, 7]
Generated pole Y offset: 196.67 mm
Run saved to: './RAW_DATA\Run101\Bumper_Beam_AP_meshed_0000.rad'
Mapping database
No files found..
Please check file stem:
e.g
C:/Folder/ModelA00*
Would be:
C:/Folder/Model*
Run 2 completed!

Run 3/3: ./RAW_DATA\Run102
Generated thickness: 1.2037 mm for props: [2, 7]
Generated pole Y offset: -257.81 mm
Run saved to: './RAW_DATA\Run102\Bumper_Beam_AP_meshed_0000.rad'
Mapping database
No files found..
Please check file stem:
e.g
C:/Folder/ModelA00*